In [ ]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="keras.src.export.tf2onnx_lib"
)
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.constants import *
from src.helpers import load_img
from sklearn.model_selection import train_test_split

ImportError: cannot import name 'VAL_SIZE' from 'src.constants' (/home/silver/cv/cyrillOCR/src/constants.py)

### File paths and labels

In [5]:
df = pd.read_csv(TRAIN_TSV, sep='\t', header=None, names=['file', 'label'])
df = df.dropna(subset=['label']).reset_index(drop=True) # 2 entries with no label
image_paths = [TRAIN_DIR + "/" + f for f in df["file"].values]
labels = df["label"].values

In [9]:
def build_dataset(paths, labels, batch_size, training=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=1024, reshuffle_each_iteration=True)

    ds = ds.map(
        lambda p, y: (load_img(p), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

### Augmentation

In [7]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(
        factor= 4 / 180.0,
        fill_mode="constant",
        fill_value=0.0
    ),
    tf.keras.layers.RandomZoom(
        height_factor=0.0,
        width_factor=(-0.1, 0.1),
        fill_mode="constant",
        fill_value=0.0
    ),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
])

I0000 00:00:1768580387.596484    2978 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6053 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


### Alphabet of possible values (lowercase only)

In [34]:
chars = set(['а','б','в','г','д','е','ж','з','и','й','к','л','м','н',
            'о','п','р','с','т','у','ф','х','ц','ч','ш','щ','ъ','ы',
            'ь','э','ю','я', 'BLANK', '1', '2', '3', '4', '5', '6',
            '7', '8', '9', '0', ' ', '!', '"', '%', '(', ')', '-',
            '.', ',', 'EOS'])

In [ ]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=VALIDATION_SIZE, random_state=67)

train_ds = build_dataset(train_paths, train_labels, batch_size=32, training=True)
val_ds = build_dataset(val_paths, val_labels, batch_size=32, training=False)